# 03 — Export for the web map
Turn each geohash into a rectangle and save shapes + daily values as one GeoJSON file.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROCESSED = Path("..") / "data" / "processed"
WEB_DATA = Path("..") / "docs" / "data"
WEB_DATA.mkdir(parents=True, exist_ok=True)

In [2]:
progress = pd.read_parquet(PROCESSED / "progress.parquet")
cells = pd.read_csv(PROCESSED / "cells.csv", index_col="GEOHASH")

DAYS = sorted(progress["date"].unique())
print(progress.shape, cells.shape, len(DAYS))

(50526, 8) (2406, 6) 21


In [3]:
BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def geohash_bbox(gh):
    lat = [-90.0, 90.0]
    lon = [-180.0, 180.0]
    use_lon = True
    for ch in gh:
        bits = BASE32.index(ch)
        for mask in (16, 8, 4, 2, 1):
            rng = lon if use_lon else lat
            mid = (rng[0] + rng[1]) / 2
            if bits & mask:
                rng[0] = mid
            else:
                rng[1] = mid
            use_lon = not use_lon
    return lat[0], lat[1], lon[0], lon[1]

geohash_bbox("sxk96p")

(41.0394287109375, 41.044921875, 28.916015625, 28.927001953125)

In [4]:
boxes = pd.DataFrame([geohash_bbox(g) for g in cells.index],
                     index=cells.index, columns=["south", "north", "west", "east"])

centre_lat = (boxes["south"] + boxes["north"]) / 2
centre_lon = (boxes["west"] + boxes["east"]) / 2

print("Max latitude difference:", (centre_lat - cells["lat"]).abs().max())
print("Max longitude difference:", (centre_lon - cells["lon"]).abs().max())

Max latitude difference: 4.973799150320701e-14
Max longitude difference: 0.0


In [5]:
def wide(col):
    return progress.pivot(index="geohash", columns="t", values=col).reindex(cells.index)

mean_w = wide("mean")
half_w = wide("half")
state_w = wide("state")
setA_w = wide("settled_A")

print(mean_w.shape, state_w.shape)

(2406, 21) (2406, 21)


In [6]:
def clean_list(row, digits=None):
    out = []
    for v in row:
        if pd.isna(v):
            out.append(None)
        elif digits is None:
            out.append(int(v))
        else:
            out.append(round(float(v), digits))
    return out

features = []
for gh in cells.index:
    s, n, w, e = boxes.loc[gh]
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[w, s], [e, s], [e, n], [w, n], [w, s]]],
        },
        "properties": {
            "id": gh,
            "final": None if pd.isna(cells.loc[gh, "final_mean"]) else round(float(cells.loc[gh, "final_mean"]), 1),
            "mean": clean_list(mean_w.loc[gh], 1),
            "half": clean_list(half_w.loc[gh], 2),
            "state": clean_list(state_w.loc[gh]),
            "settledA": clean_list(setA_w.loc[gh].astype(int)),
        },
    })

geojson = {"type": "FeatureCollection", "days": DAYS, "features": features}
print(len(features))

2406


In [7]:
out = WEB_DATA / "cells.geojson"
with open(out, "w") as f:
    json.dump(geojson, f, separators=(",", ":"))

print("Saved:", out.resolve())
print("Size: ", round(out.stat().st_size / 1e6, 2), "MB")

example = next(f for f in features if f["properties"]["id"] == "sxk691")
print(json.dumps(example["geometry"]))
print("state:", example["properties"]["state"])

Saved: /Users/saynasarvar/city-focus/web/data/cells.geojson
Size:  1.47 MB
{"type": "Polygon", "coordinates": [[[28.5205078125, 41.2261962890625], [28.531494140625, 41.2261962890625], [28.531494140625, 41.231689453125], [28.5205078125, 41.231689453125], [28.5205078125, 41.2261962890625]]]}
state: [1, 1, 1, 1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2]
